# Faithfulness Tier 2 — Local Judge (Qwen3-4B)

Chạy faithfulness Tier 2 trên `results_graphrag_final1` bằng model cục bộ,
hoàn toàn độc lập khỏi Gemini/Anthropic.

Hai model:
1. **Base**: `Qwen3-4B-Instruct-2507-Q4_K_M.gguf`
2. **Finetune**: `ft04-5k-2ep-20260729-2122-Q4_K_M.gguf`

**Cấu hình panel phải:** Accelerator `GPU T4 x2` · Internet `On` ·
Persistence `Files only` · Visibility `Private`

**Kaggle Secret bắt buộc:** `HF_TOKEN`

In [ ]:
# Cell 1 — Clone repo + install deps
import os, subprocess
from kaggle_secrets import UserSecretsClient as S
s = S()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

REPO_DIR = "/kaggle/working/repo"
BRANCH = "dev/fine-tune"
REPO_URL = "https://github.com/tandat-dao/vn-legal-graphrag.git"

# Clone
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch -q --all && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log -1 --format='commit %H%n  %s'
print(">>> GHI COMMIT HASH NÀY")

In [ ]:
# Cell 2 — Install llama-cpp-python (CUDA) + download GGUF
!pip install -q llama-cpp-python huggingface_hub hf_transfer

from huggingface_hub import hf_hub_download

HF_REPO = "dangnguyen254/thesis-graphrag-gguf"
GGUF_DIR = "/kaggle/working/models"
os.makedirs(GGUF_DIR, exist_ok=True)

BASE_GGUF = "Qwen3-4B-Instruct-2507-Q4_K_M.gguf"
FT_GGUF = "ft04-5k-2ep-20260729-2122-Q4_K_M.gguf"

for fname in [BASE_GGUF, FT_GGUF]:
    path = os.path.join(GGUF_DIR, fname)
    if not os.path.exists(path):
        print(f"Downloading {fname}...")
        hf_hub_download(
            repo_id=HF_REPO,
            filename=f"gguf/{fname}",
            local_dir=GGUF_DIR,
            local_dir_use_symlinks=False,
        )
        # hf_hub_download puts it in gguf/ subfolder
        downloaded = os.path.join(GGUF_DIR, "gguf", fname)
        if os.path.exists(downloaded) and not os.path.exists(path):
            os.rename(downloaded, path)
    print(f"  {fname}: {os.path.getsize(path) / 1e9:.2f} GB")

print("Done.")

In [ ]:
# Cell 3 — Load results + define judge logic
import json, re, time, sys
from pathlib import Path

# --- Load results ---
RESULTS_FILE = Path(REPO_DIR) / "data/evaluation/results_graphrag_final1_20260729-022916.json"
assert RESULTS_FILE.exists(), f"File not found: {RESULTS_FILE}"
with open(RESULTS_FILE, encoding="utf-8") as f:
    results_data = json.load(f)

# results_data can be dict with 'results' key or list
if isinstance(results_data, dict):
    items = results_data.get("results", results_data.get("items", []))
else:
    items = results_data

print(f"Loaded {len(items)} items from {RESULTS_FILE.name}")

# Count usable items (have context + citations)
usable = [it for it in items if it.get("context") and it.get("pred_citations")]
print(f"Usable for Tier 2: {len(usable)} (have context + citations)")
total_cits = sum(len(it.get("pred_citations", [])) for it in usable)
print(f"Total citations to judge: {total_cits}")

In [ ]:
# Cell 4 — Faithfulness judge functions
# Reuse exact same prompts from src/evaluation/faithfulness.py

JUDGE_SYSTEM_PROMPT = """Bạn là chuyên gia thẩm định citation pháp luật. Nhiệm vụ: kiểm tra xem một citation trong câu trả lời có được context (đoạn văn bản pháp luật được trích) THỰC SỰ support không.

Quy tắc đánh giá:
- SUPPORTED: nội dung claim quanh citation được context chunk này khẳng định (đồng nghĩa hoặc trích dẫn nguyên văn). Số liệu, danh từ, điều kiện đều khớp.
- UNSUPPORTED: claim KHÔNG được context chunk này khẳng định, hoặc context nói khác (số liệu khác, đối tượng khác, điều kiện khác).
- PARTIAL: context support một phần (vd: đúng đối tượng nhưng sai số liệu) — coi như UNSUPPORTED để strict.

Output JSON ONLY:
{"verdict": "SUPPORTED"|"UNSUPPORTED", "reason": "<1-2 câu giải thích ngắn>"}"""


def norm(s):
    if s is None: return None
    s = str(s).strip().lower()
    return s or None


def citation_in_context(citation, context):
    """Tier 1: citation tồn tại trong context không?"""
    vb = norm(citation.get("van_ban"))
    dieu = norm(citation.get("dieu"))
    khoan = norm(citation.get("khoan"))
    loai = norm(citation.get("loai"))
    if not vb: return False
    if not dieu and loai != "phu_luc": return False
    is_phu_luc = loai == "phu_luc" or dieu == "_default"
    headers = re.findall(r"^---[^\n]*$", context, flags=re.MULTILINE)
    for h in headers:
        h_low = h.lower()
        if vb not in h_low: continue
        if is_phu_luc:
            if "phụ lục" not in h_low: continue
            if dieu and dieu != "_default" and f"phụ lục {dieu}" not in h_low: continue
        else:
            if f"điều {dieu}." not in h_low: continue
        if khoan is not None and f"khoản {khoan}." not in h_low: continue
        return True
    return False


def extract_chunk(citation, context):
    """Lấy chunk text tương ứng citation từ context."""
    vb = norm(citation.get("van_ban"))
    dieu = norm(citation.get("dieu"))
    khoan = norm(citation.get("khoan"))
    diem = norm(citation.get("diem"))
    loai = norm(citation.get("loai"))
    if not vb: return None
    if not dieu and loai != "phu_luc": return None
    is_phu_luc = loai == "phu_luc" or dieu == "_default"
    blocks = re.split(r"(?=^---[^\n]*$)", context, flags=re.MULTILINE)
    
    def _match(block, with_diem):
        first_line_end = block.find("\n")
        header = (block[:first_line_end] if first_line_end > 0 else block).lower()
        if vb not in header: return False
        if is_phu_luc:
            if "phụ lục" not in header: return False
            if dieu and dieu != "_default" and f"phụ lục {dieu}" not in header: return False
        else:
            if f"điều {dieu}." not in header: return False
        if khoan is not None and f"khoản {khoan}." not in header: return False
        if with_diem and diem is not None and f"điểm {diem}." not in header: return False
        return True
    
    for with_diem in (True, False):
        if with_diem and diem is None: continue
        for block in blocks:
            if _match(block, with_diem): return block.strip()
    return None


def extract_answer_snippet(citation, answer, window=200):
    """Lấy đoạn answer quanh citation."""
    dieu = citation.get("dieu", "") or ""
    khoan = citation.get("khoan", "")
    vb = citation.get("van_ban", "")
    loai = citation.get("loai")
    vb_esc = re.escape(vb)
    patterns = []
    if loai == "phu_luc":
        head = r"\[Phụ\s*lục"
        if dieu and dieu != "_default": head += rf"\s+{re.escape(dieu)}"
        if khoan: patterns.append(head + rf"[^\]]*Khoản\s+{re.escape(khoan)}[^\]]*{vb_esc}\]")
        patterns.append(head + rf"[^\]]*{vb_esc}\]")
    else:
        if khoan: patterns.append(rf"\[Điều\s+{re.escape(dieu)},\s*Khoản\s+{re.escape(khoan)}[^\]]*{vb_esc}\]")
        patterns.append(rf"\[Điều\s+{re.escape(dieu)}[^\]]*{vb_esc}\]")
    for pat in patterns:
        m = re.search(pat, answer, re.IGNORECASE)
        if m:
            start = max(0, m.start() - window)
            end = min(len(answer), m.end() + 50)
            return answer[start:end]
    return answer[:window * 2]


def build_judge_prompt(citation, answer, chunk):
    """Dựng user prompt cho judge — giống hệt faithfulness.py."""
    snippet = extract_answer_snippet(citation, answer)
    return f"""CITATION cần đánh giá: {citation}

ĐOẠN ANSWER (xung quanh citation):
{snippet}

CONTEXT CHUNK được cite:
{chunk[:1500]}

Đoạn answer trên cite chunk này có được chunk THỰC SỰ support không?"""


def parse_verdict(raw):
    """Parse JSON verdict từ model output — 3 tầng fallback giống faithfulness.py."""
    raw = raw.strip()
    # Strip markdown fences
    if raw.startswith("```"):
        raw = raw.split("```", 2)[1].lstrip("json").strip()
    # Try JSON parse
    try:
        parsed = json.loads(raw)
        return parsed.get("verdict", "").upper() == "SUPPORTED", parsed.get("reason", "")
    except json.JSONDecodeError:
        pass
    # Regex fallback
    verdict_m = re.search(r'"verdict"\s*:\s*"(SUPPORTED|UNSUPPORTED)"', raw, re.IGNORECASE)
    if verdict_m:
        reason_m = re.search(r'"reason"\s*:\s*"([^"]*)', raw)
        return verdict_m.group(1).upper() == "SUPPORTED", reason_m.group(1) if reason_m else raw[:200]
    # Last resort: keyword
    if "SUPPORTED" in raw.upper() and "UNSUPPORTED" not in raw.upper():
        return True, raw[:200]
    return False, raw[:200]

print("Judge functions loaded.")

In [ ]:
# Cell 5 — Run faithfulness with a given GGUF model
from llama_cpp import Llama

def run_faithfulness(model_path, model_label, items, max_items=None):
    """Chạy Tier 1 + Tier 2 faithfulness trên items bằng local GGUF model."""
    print(f"\n{'='*60}")
    print(f"Judge: {model_label}")
    print(f"Model: {os.path.basename(model_path)}")
    print(f"{'='*60}")
    
    # Load model
    print("Loading model...")
    llm = Llama(
        model_path=model_path,
        n_ctx=4096,       # judge prompt ngắn, không cần 16k
        n_gpu_layers=-1,  # toàn bộ lên GPU
        verbose=False,
    )
    print("Model loaded.")
    
    usable = [it for it in items if it.get("context") and it.get("pred_citations")]
    if max_items:
        usable = usable[:max_items]
    
    all_results = []
    n_existing = 0
    n_supported = 0
    n_total = 0
    n_errors = 0
    t0 = time.time()
    
    for i, item in enumerate(usable):
        qid = item.get("id", f"Q{i}")
        context = item["context"]
        answer = item.get("answer", "")
        citations = item["pred_citations"]
        
        for c in citations:
            n_total += 1
            exists = citation_in_context(c, context)
            entry = {"qid": qid, "citation": c, "existing": exists}
            
            if exists:
                n_existing += 1
                chunk = extract_chunk(c, context)
                if chunk:
                    user_prompt = build_judge_prompt(c, answer, chunk)
                    try:
                        response = llm.create_chat_completion(
                            messages=[
                                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                                {"role": "user", "content": user_prompt},
                            ],
                            max_tokens=512,
                            temperature=0,
                        )
                        raw = response["choices"][0]["message"]["content"]
                        supported, reason = parse_verdict(raw)
                        entry["supported"] = supported
                        entry["reason"] = reason
                        if supported:
                            n_supported += 1
                    except Exception as e:
                        entry["supported"] = False
                        entry["reason"] = f"judge_error: {e}"
                        n_errors += 1
                else:
                    entry["supported"] = False
                    entry["reason"] = "chunk_extract_failed"
            else:
                entry["supported"] = False
                entry["reason"] = "citation_not_in_context"
            
            all_results.append(entry)
        
        # Progress
        elapsed = time.time() - t0
        if (i + 1) % 10 == 0 or i == len(usable) - 1:
            print(f"  [{i+1}/{len(usable)}] {n_total} citations, "
                  f"{n_supported}/{n_existing} supported, "
                  f"{n_errors} errors, {elapsed:.0f}s")
    
    # Free model
    del llm
    
    # Summary
    elapsed = time.time() - t0
    existence_rate = n_existing / n_total if n_total else 0
    support_rate = n_supported / n_existing if n_existing else 0
    faithful_rate = n_supported / n_total if n_total else 0
    
    summary = {
        "judge_model": model_label,
        "gguf_file": os.path.basename(model_path),
        "citations_total": n_total,
        "citations_existing": n_existing,
        "citations_supported": n_supported,
        "judge_errors": n_errors,
        "existence_rate": round(existence_rate, 4),
        "support_rate": round(support_rate, 4),
        "faithful_rate": round(faithful_rate, 4),
        "elapsed_seconds": round(elapsed, 1),
    }
    
    print(f"\n--- {model_label} ---")
    print(f"  Existence: {n_existing}/{n_total} = {existence_rate:.1%}")
    print(f"  Support:   {n_supported}/{n_existing} = {support_rate:.1%}")
    print(f"  Faithful:  {n_supported}/{n_total} = {faithful_rate:.1%}")
    print(f"  Errors:    {n_errors}")
    print(f"  Time:      {elapsed:.0f}s")
    if n_errors > 0:
        print(f"  ⚠️  {n_errors} judge errors — kết quả có thể bị ảnh hưởng!")
    
    return {"summary": summary, "per_citation": all_results}

print("Runner loaded.")

In [ ]:
# Cell 6 — Dry run: 3 items to verify everything works
dry = run_faithfulness(
    model_path=os.path.join(GGUF_DIR, BASE_GGUF),
    model_label="Qwen3-4B-Instruct (base, dry-run)",
    items=items,
    max_items=3,
)
print("\nDry run OK. Check results above before running full.")

In [ ]:
# Cell 7 — Full run: Base model
results_base = run_faithfulness(
    model_path=os.path.join(GGUF_DIR, BASE_GGUF),
    model_label="Qwen3-4B-Instruct-2507 (base)",
    items=items,
)

# Save
out_path = "/kaggle/working/faithfulness_qwen3_base.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results_base, f, ensure_ascii=False, indent=2)
print(f"\nSaved to {out_path}")

In [ ]:
# Cell 8 — Full run: Finetuned model
results_ft = run_faithfulness(
    model_path=os.path.join(GGUF_DIR, FT_GGUF),
    model_label="Qwen3-4B-FT (ft04-5k-2ep)",
    items=items,
)

# Save
out_path = "/kaggle/working/faithfulness_qwen3_ft.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results_ft, f, ensure_ascii=False, indent=2)
print(f"\nSaved to {out_path}")

In [ ]:
# Cell 9 — Comparison table
print("\n" + "="*70)
print("BẢNG SO SÁNH FAITHFULNESS TIER 2 — 4 JUDGE")
print("="*70)
print(f"{'Judge':<35} {'Loại':<25} {'Support Rate'}")
print("-"*70)
print(f"{'Gemini 2.5 Pro':<35} {'Cùng model sinh':<25} {'260/295 = 88.1% (báo cáo)'}")
print(f"{'Claude Haiku 4.5':<35} {'Khác nhà':<25} {'260/295 = 88.1% (V3)'}")
print(f"{'Gemini 2.5 Flash':<35} {'Cùng nhà, khác model':<25} {'230/295 = 78.0% (V3)'}")

sb = results_base["summary"]
sf = results_ft["summary"]

print(f"{'Qwen3-4B Base':<35} {'Open-source, độc lập':<25} "
      f"{sb['citations_supported']}/{sb['citations_existing']} = {sb['support_rate']:.1%}")
print(f"{'Qwen3-4B Finetune':<35} {'Open-source, pháp luật VN':<25} "
      f"{sf['citations_supported']}/{sf['citations_existing']} = {sf['support_rate']:.1%}")
print("-"*70)
print(f"\nBase errors: {sb['judge_errors']} | FT errors: {sf['judge_errors']}")
print(f"Base time: {sb['elapsed_seconds']:.0f}s | FT time: {sf['elapsed_seconds']:.0f}s")

In [ ]:
# Cell 10 — Upload results to HF (optional)
from huggingface_hub import HfApi
api = HfApi()

for fname in ["faithfulness_qwen3_base.json", "faithfulness_qwen3_ft.json"]:
    fpath = f"/kaggle/working/{fname}"
    if os.path.exists(fpath):
        api.upload_file(
            path_or_fileobj=fpath,
            path_in_repo=f"faithfulness/{fname}",
            repo_id=HF_REPO,
            repo_type="model",
        )
        print(f"Uploaded {fname}")

print("Done. Download from HF or copy output above.")